# Encrypted Quipu Test 52 — ECIES single-key sender, three decode paths

**Sender (payer + ECIES author)**: apocrypha. Three envelopes, three decode paths: sender self, single-key recipient (test1), multikey aggregate (test1+test2+test3 = `test_multisig3` identity).

## How this protocol works

ECIES (Elliptic Curve Integrated Encryption Scheme) is a **hybrid** scheme: each recipient gets a per-recipient "envelope" containing a shared session key, and the actual body is encrypted with that session key. Hybrid because the asymmetric crypto (ECDH) is only used to wrap the symmetric key (AES), not the content itself — keeps the body fast to decrypt and a constant size regardless of recipient count.

**The crypto, step by step:**

**1. Session key.** A fresh random 32-byte AES key is generated per inscription:
```
session_key = random_256_bits()
```

**2. Per-recipient envelopes.** For each recipient pubkey, the sender computes a shared point on secp256k1 via ECDH:
```
shared_point = sender_priv × recipient_pub        [scalar × point]
```
ECDH symmetry: the recipient computes the same point as `recipient_priv × sender_pub`.
The shared point is HKDF'd to a 32-byte envelope key, which AES-CBC-wraps the session key.
Each envelope is exactly 64 bytes (16 B IV + 48 B AES-CBC of 32-byte session).

**3. Length-prefixed inner framing** (same as AES sub-family):
```
framed = <header_len:2 BE> <inner_header> <inner_body>
```

**4. Body encryption** with the session key:
```
ciphertext = AES-CBC(session_key, framed)
```

**5. Outer header**:
```
c1dd 0001  0e  <tone>  ec  <variant=0x00>  [|TITLE|]
```

**6. Body**: `<Nrec:1>  N × <envelope:64>  <ciphertext>`

**7. Three decode paths in this notebook**:

| envelope | recipient identity | how to decrypt |
|---|---|---|
| 0 | sender's own pubkey (`pub_apo`) | apocrypha's privkey + apocrypha's own pubkey |
| 1 | single-key (`pub_test1`) | `priv_test1` + apocrypha's pubkey |
| 2 | multikey aggregate of `test1+test2+test3` | reconstruct aggregate privkey (all three cosigners) + apocrypha's pubkey |

**Author pubkey recovery.** Critical for ECIES. The sender's pubkey is recovered from the cabeza tx's input scriptSig (`<sig> <pubkey>` for P2PKH spends).

**Tamper check.** After decryption, `inner_header[:4] == c1dd 0001` confirms key correctness.

**Fee scaling**: uses `scaled_fee()` for root and join txs (1 DOGE/KB target). Wait cells after each broadcast.

**Payer**: apocrypha (single-key, mi_prv).

## Setup

In [1]:
import warnings
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL')

import os, sys, json, time, secrets
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

import cryptos
import colegio_tools as ct
from colegio_tools import _txid_of_serial
from text import build_text_quipu, read_text_quipu
from encrypted import (build_aes_quipu, build_ecies_quipu, build_keydrop_quipu,
                       read_encrypted_quipu, aggregate_privkey, aggregate_pubkey,
                       TONE_ORDINARY, TONE_AFFECTION, TONE_REVERENCE)
from coincurve import PrivateKey as CCPriv, PublicKey as CCPub

doge = cryptos.Doge()
TIP_SINGLE = 5_000_000     # 0.05 DOGE per knot for single-key strand txs
TIP_MULTI  = 10_000_000    # 0.10 DOGE per knot for multisig strand txs (size-matched to 0.2 DOGE/KB)
FEE_PER_KB = 20_000_000   # 0.2 DOGE/KB target for root + join txs

def scaled_fee(draft_hex_str, floor_sat):
    """Compute fee from drafted signed-tx size at FEE_PER_KB, floor at given TIP."""
    size_bytes = len(draft_hex_str) // 2
    return max(floor_sat, (size_bytes * FEE_PER_KB) // 1000)

In [2]:
LLAVES = os.path.abspath('../../cinv/llaves')
INSCRIPTIONS_READY = os.path.join(REPO, 'inscriptions_ready')
os.makedirs(INSCRIPTIONS_READY, exist_ok=True)

def load_priv(name, password=''):
    enc = open(os.path.join(LLAVES, f'{name}_prv.enc'), 'rb').read()
    return ct.import_privKey_from_bytes(enc, password)

# Sender (payer + ECIES author identity)
priv_apo = load_priv('mi')
addr_apo = doge.privtoaddr(priv_apo.to_hex()[2:])

# Recipients for ECIES — we use the three test keys' pubkeys (and their aggregate)
priv_test1 = load_priv('test1')
priv_test2 = load_priv('test2')
priv_test3 = load_priv('test3')

def cc_priv(p): return CCPriv(bytes.fromhex(p.to_hex()[2:]))
def cc_pub_from_priv(p): return cc_priv(p).public_key

pub_test1 = cc_pub_from_priv(priv_test1)
pub_test2 = cc_pub_from_priv(priv_test2)
pub_test3 = cc_pub_from_priv(priv_test3)

print(f'apocrypha payer: {addr_apo}')
print(f'recipients: apo (self), test1 (single-key), test1+test2+test3 (multikey aggregate)')

apocrypha payer: D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX
recipients: apo (self), test1 (single-key), test1+test2+test3 (multikey aggregate)


## Inner content + recipients

In [3]:
inner_h, inner_b = build_text_quipu('Para tres tipos de destinatarios',
                                    'Esta carta tiene tres sobres: a mi mismo, a test1, y al agregado.',
                                    tone=TONE_ORDINARY)
sender_priv = cc_priv(priv_apo)
sender_pub  = sender_priv.public_key
ms3_agg_pub = aggregate_pubkey([pub_test1, pub_test2, pub_test3])
recipients = [sender_pub, pub_test1, ms3_agg_pub]
labels = ['sender self (apocrypha)',
          'single-key recipient (test1)',
          'multikey aggregate (test1+test2+test3 = test_multisig3 identity)']
print(f'{len(recipients)} envelopes:')
for i, (p, lbl) in enumerate(zip(recipients, labels)):
    print(f'  {i}: {p.format().hex()[:20]}…  {lbl}')

3 envelopes:
  0: 037c88e9a4df6e9f4565…  sender self (apocrypha)
  1: 02f9e751cab9a9503c26…  single-key recipient (test1)
  2: 02ff87500227b1d61b00…  multikey aggregate (test1+test2+test3 = test_multisig3 identity)


## Wrap with ECIES; capture session key for keydrop

In [4]:
from coincurve.utils import get_valid_secret
import ecies as _ecies
from encrypted import _shared_key, _frame_inner, MAGIC, TYPE_ENCRYPTED, SUB_ECIES, ECIES_BROADCAST, TONE_ORDINARY
session_key = get_valid_secret()
header = MAGIC + bytes([TYPE_ENCRYPTED, TONE_ORDINARY, SUB_ECIES, ECIES_BROADCAST])
header += b'|carta cifrada a tres|'
envelopes = b''
for pub in recipients:
    envelopes += _ecies.sym_encrypt(_shared_key(sender_priv, pub), session_key)
framed = _frame_inner(inner_h, inner_b)
ciphertext = _ecies.sym_encrypt(session_key, framed)
outer_h = header
outer_b = bytes([len(recipients)]) + envelopes + ciphertext
print(f'outer ({len(outer_h)+len(outer_b)} B); session key hex: {session_key.hex()}')
SESSION_KEY_PATH = os.path.join(INSCRIPTIONS_READY, 'ecies_single_session.bin')
with open(SESSION_KEY_PATH, 'wb') as f: f.write(session_key)
print(f'session key saved to {SESSION_KEY_PATH}')

outer (362 B); session key hex: c053ff89ee7a2e0b55d019fd5a35cf37a259698d6e7aa24a3ea6784daaba5866
session key saved to /Users/anthonyschultz/Desktop/Colegio_Invisible/inscriptions_ready/ecies_single_session.bin


## Inscribe — split into strands + broadcast root

In [5]:
N_BODY_STRANDS = 4
chunk = len(outer_b) // N_BODY_STRANDS
extra = len(outer_b) %  N_BODY_STRANDS
body_parts, i = [], 0
for k in range(N_BODY_STRANDS):
    sz = chunk + (1 if k < extra else 0)
    body_parts.append(outer_b[i:i+sz]); i += sz
strand_payloads = [outer_h] + body_parts
print(f'{len(strand_payloads)} strands; sizes: {[len(p) for p in strand_payloads]}')

5 strands; sizes: [30, 83, 83, 83, 83]


In [6]:
utxos = ct.rpc_request('listunspent', [0, 9999999, [addr_apo]])
seed_inputs = [{'output': f"{u['txid']}:{u['vout']}", 'value': int(round(u['amount']*1e8))} for u in utxos]
total = sum(s['value'] for s in seed_inputs)
print(f'{len(seed_inputs)} UTXO(s), total {total/1e8:.4f} DOGE')

1 UTXO(s), total 12.7896 DOGE


### Build root tx with scaled fee

In [7]:
priv_hex = priv_apo.to_hex()[2:]
n = len(strand_payloads)

# Draft pass — placeholder seeds with TIP fee to measure tx size
draft_per = (total - TIP_SINGLE) // n
draft_seeds = [draft_per] * n
draft = doge.mktx(seed_inputs, [{'value': s, 'address': addr_apo} for s in draft_seeds])
doge.signall(draft, priv_hex)
draft_hex = cryptos.serialize(draft)
root_fee = scaled_fee(draft_hex, TIP_SINGLE)
print(f'root draft size: {len(draft_hex)//2} B  ->  scaled fee: {root_fee/1e8:.4f} DOGE')

# Real pass — redistribute strand seeds with correct fee
per = (total - root_fee) // n
remainder = (total - root_fee) - per * n
strand_seeds = [per] * n
strand_seeds[0] += remainder
root_outputs = [{'value': s, 'address': addr_apo} for s in strand_seeds]
root_tx = doge.mktx(seed_inputs, root_outputs)
doge.signall(root_tx, priv_hex)
root_hex = cryptos.serialize(root_tx)
root_txid = _txid_of_serial(root_hex)
assert ct.rpc_request('sendrawtransaction', [root_hex]) == root_txid
print(f'root_txid: {root_txid}')

root draft size: 360 B  ->  scaled fee: 0.0720 DOGE
root_txid: 1bbc2dffbb40e1a93467a123a9af9890d8bb37253e420b7af0392d9f239ba818


In [8]:
# Wait for root to confirm
print(f'waiting for root to confirm... ({root_txid[:16]}…)')
start_h = ct.rpc_request('getblockcount')
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        info = ct.rpc_request('getrawtransaction', [root_txid, 1])
        confs = info.get('confirmations', 0)
        print(f'  {time.strftime("%H:%M:%S")}  block {h}  root confs: {confs}')
        if confs >= 1:
            print(f'✓ root confirmed in block {info.get("blockhash","?")}')
            break
        start_h = h
    time.sleep(15)

waiting for root to confirm... (1bbc2dffbb40e1a9…)
  20:08:17  block 6213638  root confs: 0
  20:09:02  block 6213640  root confs: 0
  20:10:02  block 6213641  root confs: 1
✓ root confirmed in block e9a808521a14d6ba01ac24408a8157d1c4801f1f9c2ed9f9efe3f3fffaff8887


### Phase II — precompute + broadcast strand chains

In [9]:
strands = []
for si, payload in enumerate(strand_payloads):
    cad = ct.CadenaAtom(prvkey=priv_hex, data=payload,
                         utxo_dct={'output': f'{root_txid}:{si}', 'value': strand_seeds[si]},
                         tip=TIP_SINGLE)
    cad.precompute()
    strands.append(cad)
    print(f'  strand {si}: {len(cad.txns)} knots')
for si, cad in enumerate(strands):
    for hex_tx, txid in zip(cad.txns, cad.txn_ids):
        assert ct.rpc_request('sendrawtransaction', [hex_tx]) == txid
    print(f'  strand {si} broadcast')

  strand 0: 1 knots
  strand 1: 2 knots
  strand 2: 2 knots
  strand 3: 2 knots
  strand 4: 2 knots
  strand 0 broadcast
  strand 1 broadcast
  strand 2 broadcast
  strand 3 broadcast
  strand 4 broadcast


In [10]:
# Wait for all strand termini to confirm
print('waiting for strand termini to confirm...')
start_h = ct.rpc_request('getblockcount')
termini = [c.txn_ids[-1] for c in strands]
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        confs = [ct.rpc_request('getrawtransaction', [t, 1]).get('confirmations', 0) for t in termini]
        print(f'  block {h}  ' + '  '.join(f's{i}:{c}' for i,c in enumerate(confs)))
        if all(c >= 1 for c in confs):
            print('✓ all strand termini confirmed')
            break
        start_h = h
    time.sleep(15)

waiting for strand termini to confirm...
  block 6214467  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214468  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214469  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214470  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214471  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214473  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214474  s0:1  s1:1  s2:1  s3:1  s4:1
✓ all strand termini confirmed


### Phase III — build join tx with scaled fee

In [11]:
join_inputs = [{'output': f'{c.txn_ids[-1]}:0',
                'value': strand_seeds[si] - TIP_SINGLE * len(c.txns)}
               for si, c in enumerate(strands)]
join_total = sum(i['value'] for i in join_inputs)

# Draft pass
draft = doge.mktx(join_inputs, [{'value': join_total - TIP_SINGLE, 'address': addr_apo}])
doge.signall(draft, priv_hex)
draft_hex = cryptos.serialize(draft)
join_fee = scaled_fee(draft_hex, TIP_SINGLE)
print(f'join draft size: {len(draft_hex)//2} B  ->  scaled fee: {join_fee/1e8:.4f} DOGE')

# Real pass
join_tx = doge.mktx(join_inputs, [{'value': join_total - join_fee, 'address': addr_apo}])
doge.signall(join_tx, priv_hex)
join_hex = cryptos.serialize(join_tx)
join_txid = _txid_of_serial(join_hex)
assert ct.rpc_request('sendrawtransaction', [join_hex]) == join_txid
print(f'join_txid: {join_txid}')

join draft size: 942 B  ->  scaled fee: 0.1884 DOGE
join_txid: 1a5f1e3cd32ae538132198b48743282ae38f0281c8ea93c578d9a9bdab7ed891


In [12]:
# Wait for join to confirm
print(f'waiting for join to confirm... ({join_txid[:16]}…)')
start_h = ct.rpc_request('getblockcount')
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        info = ct.rpc_request('getrawtransaction', [join_txid, 1])
        confs = info.get('confirmations', 0)
        print(f'  {time.strftime("%H:%M:%S")}  block {h}  join confs: {confs}')
        if confs >= 1:
            print(f'✓ join confirmed in block {info.get("blockhash","?")}')
            break
        start_h = h
    time.sleep(15)

waiting for join to confirm... (1a5f1e3cd32ae538…)
  10:53:40  block 6214475  join confs: 0
  10:54:25  block 6214477  join confs: 2
✓ join confirmed in block 815e294d12c2035ffa7752d6dc739ab5d790aa5f1982644dbff04d5be3343f96


## Read back

In [13]:
# Walk the diamond using our local spender map
spender_map = {}
for si, cad in enumerate(strands):
    spender_map[f'{root_txid}:{si}'] = cad.txn_ids[0]
    for ki in range(len(cad.txn_ids) - 1):
        spender_map[f'{cad.txn_ids[ki]}:0'] = cad.txn_ids[ki+1]
def walk(start):
    out, cur = '', start
    while True:
        n = spender_map.get(cur)
        if not n: return out
        raw = ct.rpc_request('getrawtransaction', [n, 1])
        op = next((ct.extract_op_return(v) for v in raw['vout'] if ct.extract_op_return(v)), None)
        if not op: return out
        out += op; cur = f'{n}:0'
rec_h = bytes.fromhex(walk(f'{root_txid}:0'))
rec_b = b''.join(bytes.fromhex(walk(f'{root_txid}:{si}')) for si in range(1, len(strands)))
assert rec_h == outer_h and rec_b == outer_b
print('✓ recovered byte-identical')

✓ recovered byte-identical


## Decode path 1 — sender self (apocrypha)

In [14]:
parsed_self = read_encrypted_quipu(rec_h, rec_b,
                                     my_privkey=sender_priv,
                                     author_pubkey=sender_pub)
assert parsed_self['inner_body'] == inner_b
print('✓ apocrypha self-decrypted (envelope 0)')

✓ apocrypha self-decrypted (envelope 0)


## Decode path 2 — single-key recipient (test1)

In [15]:
parsed_single = read_encrypted_quipu(rec_h, rec_b,
                                       my_privkey=cc_priv(priv_test1),
                                       author_pubkey=sender_pub)
assert parsed_single['inner_body'] == inner_b
print('✓ test1 decrypted via envelope 1')

✓ test1 decrypted via envelope 1


## Decode path 3 — multikey aggregate (test_multisig3)

In [16]:
agg_priv_ms3 = aggregate_privkey([cc_priv(priv_test1), cc_priv(priv_test2), cc_priv(priv_test3)])
assert agg_priv_ms3.public_key.format() == ms3_agg_pub.format()
parsed_multi = read_encrypted_quipu(rec_h, rec_b,
                                      my_privkey=agg_priv_ms3,
                                      author_pubkey=sender_pub)
assert parsed_multi['inner_body'] == inner_b
print('✓ test_multisig3 aggregate decrypted via envelope 2')
print('  (required ALL THREE test cosigners to cooperate)')

✓ test_multisig3 aggregate decrypted via envelope 2
  (required ALL THREE test cosigners to cooperate)


In [18]:
# === Show decrypted payload for each recipient ===
from text import read_text_quipu
from encrypted import read_encrypted_quipu, aggregate_privkey

# Each recipient + how they unwrap their envelope.
decoders = [
    ("sender self (apocrypha)",
     cc_priv(priv_apo),                                          # privkey
     sender_pub),                                                # author pubkey (self)
    ("single-key recipient (test1)",
     cc_priv(priv_test1),
     sender_pub),
    ("multikey aggregate (test1+test2+test3)",
     aggregate_privkey([cc_priv(priv_test1), cc_priv(priv_test2), cc_priv(priv_test3)]),
     sender_pub),
]

for label, my_priv, author_pub in decoders:
    parsed = read_encrypted_quipu(outer_h, outer_b,
                                    my_privkey=my_priv,
                                    author_pubkey=author_pub)
    inner = read_text_quipu(parsed['inner_header'], parsed['inner_body'])
    print(f"--- {label} ---")
    print(f"  magic_ok : {parsed['magic_ok']}")
    print(f"  title    : {inner['title']}")
    print(f"  tone     : 0x{inner['tone']:02x}")
    print(f"  body     : {inner['body']}")
    print()

--- sender self (apocrypha) ---
  magic_ok : True
  title    : Para tres tipos de destinatarios
  tone     : 0x00
  body     : Esta carta tiene tres sobres: a mi mismo, a test1, y al agregado.

--- single-key recipient (test1) ---
  magic_ok : True
  title    : Para tres tipos de destinatarios
  tone     : 0x00
  body     : Esta carta tiene tres sobres: a mi mismo, a test1, y al agregado.

--- multikey aggregate (test1+test2+test3) ---
  magic_ok : True
  title    : Para tres tipos de destinatarios
  tone     : 0x00
  body     : Esta carta tiene tres sobres: a mi mismo, a test1, y al agregado.



## Save manifest

In [17]:
json.dump({'root_txid': root_txid, 'join_txid': join_txid,
           'session_key_path': SESSION_KEY_PATH,
           'sender': addr_apo, 'recipients': labels},
          open(os.path.join(INSCRIPTIONS_READY, 'ecies_single_manifest.json'), 'w'),
          indent=2)
print('manifest saved')

manifest saved
